In [1]:
# Imports

import pandas as pd
import os
import glob
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import kerastuner as kt

Using TensorFlow backend


C:\Users\Sairaj\AppData\Local\Temp\ipykernel_16228\583349949.py:16: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [2]:
# INPUT PARAMETERS

folder_path = r'E:\SMART GRID FOR EV\SAME_METERS\TESTING_1'  # Specify the folder where your CSV files are located

output_folder = r"E:\SMART GRID FOR EV\RESULTS\DAILY"

time_step =  21
EVALUATION_INTERVAL = 100
#EPOCHS = 20

L1 = 14
L2 = 24

LD1 = 1
Patience = 3
count = 0

In [3]:
# Function to Convert hourly columns to rows 

def col_row(df):
    df.fillna(0, inplace=True)                                                    # Filling empty datapoints with zero
    
    data_list = []                                                                # Create a new list to store the transformed data

    for _, row in df.iterrows():                                                  # Loop through each row in the original DataFrame
        for hour in range(1, 25):                                                 # Loop through each column from 3 to 26 (1 to 24)
            col_name = str(hour)                                                  # Get the column name
            value = row[col_name]                                                 # Get the value from the original DataFrame
            new_row = {'Meter_reading': value}                                    # Creating new row and assigning the value
            data_list.append(new_row)                                             # Append the new dictionary to the list

    new_df_hourly = pd.DataFrame(data_list)                                       # Convert the list of dictionaries into a new DataFrame

    new_df_hourly['Reading_Date'] = pd.date_range(start='01-01-2018 01:00:00', periods=len(new_df_hourly), freq='H')            # Adding datetime column starting from 01-01-2018 hourly till 31-12-2022
    new_df_hourly['Meter_reading'] = new_df_hourly['Meter_reading'] * 1000        # Converting Kw to Watt by *1000
    nozero_df_hourly = new_df_hourly[new_df_hourly['Meter_reading'] != 0]         # Skipping zero value datapoints and creating new dataframe

    return new_df_hourly, nozero_df_hourly

In [4]:
# Function to Extract data from csv to Dataframe

def excel_to_df(url):

    match = re.search(r'(\d+)_merged\.csv', url)                    # Extracting meter number from CSV name
    meter_number = match.group(1)

    dataframe = pd.read_csv(url)

    new_df_hourly, nozero_df_hourly = col_row(dataframe)            # Function to Convert hourly columns to hourly rows 

    df_daily = dataframe.copy()                                                 # Copying main dataframe to create daily consumption data frame
    df_daily['Meter_reading'] =  df_daily.iloc[:, 2:26].sum(axis=1)* 1000       # Adding up 24 hours datapoint values
    df_daily = df_daily.drop(df_daily.iloc[:, 1:26],axis=1)                     # Dropping hourly columns after adding

    return df_daily, new_df_hourly, nozero_df_hourly, meter_number

In [5]:
# Normalising and dividing data into Train, Val and test

def data_splitting_shapping(dataset,time_step):
    
    target_dataset = dataset["Meter_reading"]

    from sklearn.preprocessing import MinMaxScaler              # Normalize data before model fitting as it will boost the performance( in neural networks) + transform
    scaler = MinMaxScaler(feature_range = (0,1))                # scale of the output and input inthe range 0-1 to match the scale of the layer of LSTM
    target_dataset = scaler.fit_transform(np.array(target_dataset).reshape(-1,1))         # reshape: convert the univariate 1D array into 2D
    
    train_size = int(len(target_dataset)*0.80)
    test_size = len(target_dataset)- train_size
    val_size = int(train_size*0.20)
    
    train_data = target_dataset[0:train_size-val_size, : ]                # Splitting data into train test Val
    test_data = target_dataset[train_size:len(target_dataset), :1 ]
    val_data = target_dataset[len(target_dataset)-test_size-val_size:len(target_dataset)-test_size, :1 ]

    def create_dataset(dataset, time_step = 1):                 # Function to create X and Y nparray from dataset and timestep for LSTM model
        dataX, dataY = [] , []
        for i in range(len(dataset)-time_step-1):
            a = dataset[i:(i+time_step),0]
            dataX.append(a)
            dataY.append(dataset[i + time_step,0])
        return np.array(dataX), np.array(dataY)
    
    X_train, y_train = create_dataset(train_data, time_step)
    X_test, y_test = create_dataset(test_data, time_step)
    X_val, y_val = create_dataset(val_data, time_step)

    X_train = X_train.reshape(X_train.shape[0],X_train.shape[1], 1)         # Reshaping arrays according LSTM model
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1],1)
    X_val = X_val.reshape(X_val.shape[0], X_val.shape[1],1)

    BATCH_SIZE = 256
    BUFFER_SIZE = 10000

    train_univariate = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_univariate = train_univariate.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE).repeat()

    val_univariate = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_univariate = val_univariate.batch(BATCH_SIZE).repeat()
    
    return X_train, X_test, X_val, y_train, y_test, y_val, scaler, train_univariate, val_univariate

In [6]:
# LSTM Model Architecture

def lstm_archi(EVALUATION_INTERVAL, EPOCHS, time_step, X_train, y_train, X_val, y_val, train_univariate, val_univariate, L1, L2, LD1, Patience):
    
    from keras.callbacks import EarlyStopping

    early_stopping = EarlyStopping(monitor = 'val_loss', patience=Patience)                 # Define the early stopping callback

    tf.keras.backend.clear_session()
    
    simple_lstm_model = tf.keras.models.Sequential([tf.keras.layers.LSTM(L1, input_shape=(time_step, 1)),               # Model Architecture
                                                   tf.keras.layers.Dense(LD1) ])
    
    simple_lstm_model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])
    
    
    #history = simple_lstm_model.fit(X_train, y_train, epochs=EPOCHS, steps_per_epoch=EVALUATION_INTERVAL, validation_data=(X_val,y_val), validation_steps=50, callbacks=[early_stopping], verbose=0)
    
    history = simple_lstm_model.fit(train_univariate, epochs=EPOCHS, steps_per_epoch=EVALUATION_INTERVAL, validation_data=val_univariate, validation_steps=50, callbacks=[early_stopping], verbose=0)


    return simple_lstm_model, history

In [7]:
# Plotting Energy distribution and Model loss

def loss_graph(history, dataset):
    plt.figure(figsize=(10, 5))

    # Plot model loss on the left side
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')

    # Plot Energy Distribution on the right side
    plt.subplot(1, 2, 2)
    sns.distplot(dataset["Meter_reading"])
    plt.title('Energy Distribution')

    plt.tight_layout()  # Ensure proper spacing between subplots
    plt.show()


In [8]:
# Model evaluation to find MAE, MAPE, RMSE

def model_eval( simple_lstm_model, scaler, X_test, y_test):

   # Predicting consumption using test data

    test_predictions_1 = simple_lstm_model.predict(X_test)

    test_predictions =scaler.inverse_transform(test_predictions_1)

    y_test = y_test.reshape(y_test.shape[0], 1)

    actual_test = scaler.inverse_transform(y_test)

    mean_actual = np.mean(actual_test)  # Calculate the mean of the actual target variable

    # A lower RMSE, MAE, and MAPE for your model compared to the baseline indicate that your model is performing better.
    
    raw_errors = test_predictions - actual_test

    # Calculate MAE (Mean Absolute error)
    mae_test = np.mean(np.abs(test_predictions - actual_test))
    #mae_baseline = np.mean(np.abs(mean_actual - actual_test))  # Calculate MAE for the mean baseline


    # Calculate MAPE (Mean Absolute Percentage Error)
    #epsilon = 1e-10  # Small constant to avoid division by zero
    denominator = np.where(actual_test != 0, actual_test, 1)
    mape_test = np.mean(np.abs((actual_test - test_predictions) / denominator)) * 100
    #mape_test = np.mean(np.abs((actual_test - test_predictions) / (actual_test + epsilon))) * 100
    #mape_baseline = np.mean(np.abs((mean_actual - actual_test) / denominator)) * 100  # Calculate MAPE for the mean baseline


    # Calculate RMSE (Root Mean Squared Error)
    rmse_test = np.sqrt(np.mean((test_predictions - actual_test) ** 2))
    #rmse_baseline = np.sqrt(np.mean((mean_actual - actual_test) ** 2))  # Calculate RMSE for the mean baseline

    return mae_test, mape_test, rmse_test, test_predictions, actual_test, mean_actual, raw_errors

In [9]:
# Plotting function Actual and Test predictions

def plot_test_results(test_predictions, actual_test):
    # Create a range of indices for the data points
    indices = range(len(actual_test))

    # Plot the actual values and train predictions
    plt.figure(figsize=(20, 8))
    plt.plot(indices, actual_test, label='Actuals', marker='o', linestyle='-')
    plt.plot(indices, test_predictions, label='Test Predictions', marker='x', linestyle='--')

    # Set labels and title
    plt.xlabel('Data Point Index')
    plt.ylabel('Value')
    plt.title('Actuals vs. Train Predictions')

    # Add a legend
    plt.legend()

    # Show the plot
    plt.show()

def plot_raw_errors(raw_errors):
    indices = range(len(raw_errors))

    plt.figure(figsize=(20, 8))
    plt.plot(indices, raw_errors, label='ERROR', marker='o', linestyle='-')
    plt.xlabel('Data Point Index')
    plt.ylabel('Value')
    plt.title('ERROR PER PREDICTION')
    plt.legend()
    plt.show()


In [10]:
# Plotting graph call

def plot_graphs(Meter_number, mae_test, mape_test, rmse_test, history, dataset, test_predictions, actual_test, raw_errors):
    # Print the results
    print("=================================")
    print("Results of meter:- ",Meter_number)
    print("MAE:", mae_test)
    print("MAPE:", mape_test)
    print("RMSE:", rmse_test)
    print("=================================")

    #plot_Energy_Distribution(dataset)
    loss_graph(history, dataset)
    #plot_test_results(test_predictions, actual_test)
    #plot_raw_errors(raw_errors)

In [11]:
# Results to Excel

def results_OP(Results, output_folder):
    # create a DataFrame from the results
    Results_df = pd.DataFrame(Results, columns=['Meter_number','Mean', 'MAE', 'MAPE', 'RMSE', 'Time step', 'Evaluation interval', 'Epochs','L1', 'LD1', 'Patience'])

    from datetime import datetime
    current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_excel = os.path.join(output_folder, 'Results_L1_{}_timestep_{}_{}.xlsx'.format(L1, time_step, current_time))

    # custom_name = "Results"  # Replace with your desired name
    # Create a custom file name for the output CSV file
    # output_csv = os.path.join(output_folder, f'{custom_name}.csv')

    Results_df.to_excel(output_excel, index=False)                  # Save the DataFrame to a CSV file

    if os.path.exists(output_excel):
        print(f"CSV file saved to {output_excel}")
    else:
        print("Error: CSV file not saved.")


#Main code

Results = []
data_frames = []

for filename in os.listdir(folder_path):                   #Loop through the files in the folder
    if filename.endswith(".csv"):                           
        file_path = os.path.join(folder_path, filename)

        df_daily, new_df_hourly, nozero_df_hourly, Meter_number = excel_to_df(file_path)
        dataset = df_daily

        #newdataset = df_daily['Meter_reading'] 
        #data_frames.append(newdataset)
        #combined_data = pd.concat(data_frames, axis=1)  #Assuming columns represent each meter

        print("=================================")
        print("Start : - ", Meter_number)
        print("=================================")

        X_train, X_test, X_val, y_train, y_test, y_val, scaler, train_univariate, val_univariate = data_splitting_shapping(dataset, time_step)

        print(" ")
        print(" TRAINING started ")

        simple_lstm_model, history = lstm_archi(EVALUATION_INTERVAL, EPOCHS, time_step, X_train, y_train, X_val, y_val, train_univariate, val_univariate, L1, L2, LD1, Patience)
        #simple_lstm_model.summary()

        print(" ")
        print(" TRAINING finished ")
        print("=================================")
        print(" ")
        print(" Evaluating Results ")
        print(" ")

        mae_test, mape_test, rmse_test, test_predictions, actual_test, mean_actual, raw_errors = model_eval( simple_lstm_model, scaler, X_test, y_test)
        
        #plot_graphs(Meter_number, mae_test, mape_test, rmse_test, history, dataset, test_predictions, actual_test, raw_errors)

        Results.append([Meter_number, mean_actual, mae_test, mape_test, rmse_test,time_step, EVALUATION_INTERVAL, EPOCHS, L1, LD1, Patience])

        print("=================================")
        print("Done : - ", Meter_number)
        print("=================================")
        count = count + 1
        print(" ")
        print(" Number of meter done in folder: ", count)
        print(" ")

results_OP(Results, output_folder)



In [ ]:
#Main code for FOR loop

Results = []

for filename in os.listdir(folder_path):                   # Loop through the files in the folder
    if filename.endswith(".csv"):
        #for i in range(14, 71, 7):
        for i in range(10, 31, 10):
            EPOCHS =  i                           
            file_path = os.path.join(folder_path, filename)

            df_daily, new_df_hourly, nozero_df_hourly, Meter_number = excel_to_df(file_path)
            dataset = df_daily

            #newdataset = df_daily['Meter_reading'] 
            #data_frames.append(newdataset)
            #combined_data = pd.concat(data_frames, axis=1)  # Assuming columns represent each meter

            print("=================================")
            print("Start : - ", Meter_number)
            print("=================================")

            X_train, X_test, X_val, y_train, y_test, y_val, scaler, train_univariate, val_univariate = data_splitting_shapping(dataset, time_step)

            print(" ")
            print(" TRAINING started ")

            simple_lstm_model, history = lstm_archi(EVALUATION_INTERVAL, EPOCHS, time_step, X_train, y_train, X_val, y_val, train_univariate, val_univariate, L1, L2, LD1, Patience)
            #simple_lstm_model.summary()

            print(" ")
            print(" TRAINING finished ")
            print("=================================")
            print(" ")
            print(" Evaluating Results ")
            print(" ")

            mae_test, mape_test, rmse_test, test_predictions, actual_test, mean_actual, raw_errors = model_eval( simple_lstm_model, scaler, X_test, y_test)
            
            #plot_graphs(Meter_number, mae_test, mape_test, rmse_test, history, dataset, test_predictions, actual_test, raw_errors)

            Results.append([Meter_number, mean_actual, mae_test, mape_test, rmse_test, time_step, EVALUATION_INTERVAL, EPOCHS, L1, LD1, Patience])

            print("=================================")
            print("Done : - ", Meter_number)
            print("=================================")
            count = count + 1
            print(" ")
            print(" Number of meter done in folder: ", count)
            print(" ")

results_OP(Results, output_folder)

